# Solución Big Data en Databricks · Arquitectura Medallion (Bronze → Silver → Gold)
**Proyecto:** Sistema automatizado para el monitoreo de empleados (BASAV · Grupo 2) · **Responsable:** Luis Obregón Hidalgo

Este notebook completa la etapa Big Data pedida en la entrega final: no solo carga el dataset, también lo **explora (EDA)**,
lo **limpia y prepara**, lo organiza en capas **Bronze, Silver y Gold**, guarda todo como **tablas Delta** en Unity Catalog,
calcula los **indicadores** (TPA, ICT, TDT, IEP) y crea un **dashboard** de Databricks sobre la capa Gold.

| Paso | Qué hace | Operaciones PySpark | Captura |
|---|---|---|---|
| 0 | Configuración, esquemas `mon_bronze`, `mon_silver`, `mon_gold` | `CREATE SCHEMA` | S-01 |
| 1 | **Bronze**: ingesta cruda de los 6 CSV (todo texto) + metadatos de origen | `read.csv`, `withColumn`, `_metadata` | S-02 |
| 2 | **EDA**: esquema, `describe`, `summary`, nulos, distintos, distribuciones | `describe`, `summary`, `groupBy`, `agg` | S-03, S-04 |
| 3 | Diagnóstico de calidad antes de limpiar | `filter`, `try_cast`, `count` | S-05 |
| 4 | **Silver**: limpieza y preparación | `select`, `withColumnRenamed`, `cast`, `trim`, `na.drop`, `na.fill`, `dropDuplicates`, `filter`, `when` | S-06 |
| 5 | Conciliación Bronze ↔ Silver y cuarentena | `join`, `count` | S-07 |
| 6 | **Gold**: indicadores global, por mes y por trabajador-mes | `groupBy`, `agg`, `join` | S-08 |
| 7 | Validación: KPI Gold de la escala base = Power BI | — | S-09 |
| 8 | Visualizaciones (matplotlib) | `toPandas` | S-10 |
| 9 | Persistencia: tablas, comentarios e historial Delta | `COMMENT ON`, `DESCRIBE HISTORY` | S-11 |
| 10 | Dashboard de Databricks (AI/BI) sobre la capa Gold | SDK `lakeview` | S-12 |

**Fuente de datos:** los mismos CSV de EXP-02 (582 filas reales del Data Mart y 5 000 / 10 000 filas sintéticas, semilla 2026),
en `/Volumes/workspace/bd_rendimiento/vol_rendimiento/datos_base`. Si no están, el paso 1 los copia desde GitHub.
Solo se usan IDs de trabajador (sin nombres, correos ni teléfonos).

## 0 · Configuración — **CAPTURA S-01**

In [ ]:
import os, io, json, platform
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
from pyspark.sql import functions as F, Window

RESPONSABLE = "Luis Obregón Hidalgo"
CATALOGO    = "workspace"
BRONZE, SILVER, GOLD = f"{CATALOGO}.mon_bronze", f"{CATALOGO}.mon_silver", f"{CATALOGO}.mon_gold"
RUTA_DATOS  = "/Volumes/workspace/bd_rendimiento/vol_rendimiento/datos_base"
RUTA_SALIDA = "/Volumes/workspace/bd_rendimiento/vol_rendimiento/resultados_solucion"
ESCALAS     = {"base": 582, "5000": 5000, "10000": 10000}
LIMITE_PUNTUAL = "09:41:00"   # regla del Data Mart: antes de 09:41 = A tiempo
ESTADOS_ASIS  = ["A tiempo", "Tardanza"]
ESTADOS_TAREA = ["Completada", "En Progreso", "Asignada"]
REF_POWER_BI  = {"TPA": 92.10, "ICT": 56.36, "TDT": 3.44, "IEP": 75.12}
TZ = ZoneInfo("America/Lima")

try:
    NOTEBOOK = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
except Exception:
    NOTEBOOK = "BigData_Solucion_Medallion"

def ahora():
    return datetime.now(TZ).strftime("%Y-%m-%d %H:%M:%S")

def marca(captura, titulo=""):
    print(f"[{captura}] {titulo} | {ahora()} (Lima) | Responsable: {RESPONSABLE} | Notebook: {NOTEBOOK}")
    print("-" * 100)

marca("S-01", "Configuración y esquemas Medallion")
for esq, desc in ((BRONZE, "Capa Bronze: datos crudos tal como llegan (texto) + metadatos de origen"),
                  (SILVER, "Capa Silver: datos limpios, tipados y validados"),
                  (GOLD,   "Capa Gold: indicadores TPA, ICT, TDT e IEP listos para el dashboard")):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {esq} COMMENT '{desc}'")
dbutils.fs.mkdirs(RUTA_SALIDA)
print("Spark", spark.version, "| Python", platform.python_version(),
      "| Runtime", os.environ.get("DATABRICKS_RUNTIME_VERSION", "no disponible"))
display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}").filter("databaseName LIKE 'mon_%'"))

## 1 · Capa **Bronze**: ingesta cruda — **CAPTURA S-02**
Se leen los 6 CSV **sin inferir tipos** (todas las columnas como texto) para guardar el dato tal como llegó.
Se agregan columnas de trazabilidad: `_escala`, `_archivo_origen` y `_fecha_ingesta`.

In [ ]:
# Si los CSV no están en el volumen, se copian desde el repositorio del grupo (misma versión que EXP-02)
import urllib.request
URL_DATOS = "https://raw.githubusercontent.com/b-alfonso/Sistema_Remuneraciones_Reportes/main/06_Big_Data_Databricks/datos"
dbutils.fs.mkdirs(RUTA_DATOS)
ya = {f.name for f in dbutils.fs.ls(RUTA_DATOS)}
for e in ESCALAS:
    for tipo in ("asistencias", "tareas"):
        a = f"{tipo}_{e}.csv"
        if a not in ya:
            urllib.request.urlretrieve(f"{URL_DATOS}/{a}", f"{RUTA_DATOS}/{a}")
            print("copiado desde GitHub:", a)
print("Archivos en el volumen:", sorted(f.name for f in dbutils.fs.ls(RUTA_DATOS) if f.name.endswith(".csv")))

In [ ]:
marca("S-02", "Bronze: ingesta cruda")
def ingesta_bronze(tipo):
    partes = []
    for e in ESCALAS:
        df = (spark.read.format("csv").option("header", "true").option("inferSchema", "false")
              .load(f"{RUTA_DATOS}/{tipo}_{e}.csv")
              .withColumn("_escala", F.lit(e))
              .withColumn("_archivo_origen", F.col("_metadata.file_name"))
              .withColumn("_fecha_ingesta", F.current_timestamp()))
        partes.append(df)
    out = partes[0]
    for p in partes[1:]:
        out = out.unionByName(p)
    return out

bronze_asis  = ingesta_bronze("asistencias")
bronze_tarea = ingesta_bronze("tareas")
bronze_asis.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BRONZE}.asistencias")
bronze_tarea.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{BRONZE}.tareas")

bronze_asis  = spark.table(f"{BRONZE}.asistencias")
bronze_tarea = spark.table(f"{BRONZE}.tareas")
bronze_asis.printSchema()
display(spark.sql(f'''
  SELECT 'asistencias' AS tabla, _escala, _archivo_origen, COUNT(*) AS filas FROM {BRONZE}.asistencias GROUP BY ALL
  UNION ALL
  SELECT 'tareas', _escala, _archivo_origen, COUNT(*) FROM {BRONZE}.tareas GROUP BY ALL
  ORDER BY tabla, filas'''))

## 2 · Análisis exploratorio (EDA) — **CAPTURAS S-03 y S-04**
Se hace sobre la escala **base** (582 filas reales). Al final se repite el conteo de nulos para las tres escalas.

In [ ]:
marca("S-03", "EDA: describe, summary, nulos y distintos (escala base)")
a0 = bronze_asis.filter("_escala = 'base'").drop("_escala", "_archivo_origen", "_fecha_ingesta")
t0 = bronze_tarea.filter("_escala = 'base'").drop("_escala", "_archivo_origen", "_fecha_ingesta")

print("Asistencias: describe()")
display(a0.describe())
print("Tareas: summary() (incluye cuartiles)")
display(t0.summary("count", "min", "25%", "50%", "75%", "max"))

def perfil(df, nombre):
    total = df.count()
    filas = []
    for c in df.columns:
        r = df.agg(F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias("nulos"),
                   F.countDistinct(c).alias("distintos")).first()
        filas.append((nombre, c, total, int(r["nulos"]), round(100 * r["nulos"] / total, 2), int(r["distintos"])))
    return spark.createDataFrame(filas, "tabla string, columna string, filas long, nulos_o_vacios long, pct_nulos double, distintos long")

display(perfil(a0, "asistencias").unionByName(perfil(t0, "tareas")))

In [ ]:
marca("S-04", "EDA: distribuciones (escala base)")
a0t = a0.withColumn("entrada", F.to_timestamp("Fecha_Entrada"))
print("Asistencias por estado")
display(a0.groupBy("Estado_Asistencia").agg(F.count("*").alias("cantidad"),
        F.round(100 * F.count("*") / a0.count(), 2).alias("porcentaje")).orderBy(F.desc("cantidad")))
print("Tareas por estado y por ¿tiene observación?")
display(t0.groupBy("Estado_Tarea").agg(
        F.count("*").alias("cantidad"),
        F.sum(F.when(F.col("Observaciones").isNotNull(), 1).otherwise(0)).alias("con_observacion"),
        F.sum(F.when(F.upper("Observaciones").contains("FUERA DE PLAZO"), 1).otherwise(0)).alias("fuera_de_plazo"))
        .orderBy(F.desc("cantidad")))
print("Asistencias por mes y estado")
display(a0t.groupBy(F.date_format("entrada", "yyyy-MM").alias("mes"), "Estado_Asistencia").count().orderBy("mes", "Estado_Asistencia"))
print("Hora de entrada: estadísticos en minutos desde 00:00")
display(a0t.select((F.hour("entrada") * 60 + F.minute("entrada")).alias("min_entrada"))
           .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max"))
print("Registros por trabajador (ID)")
display(a0.groupBy(F.col("ID_Trabajador").cast("int").alias("ID_Trabajador")).count().orderBy("ID_Trabajador"))

## 3 · Diagnóstico de calidad antes de limpiar — **CAPTURA S-05**
Cada regla cuenta cuántas filas la incumplen en Bronze, por escala. Una regla con 0 se deja igual en el pipeline:
así el proceso queda listo para datos nuevos que sí traigan esos errores.

In [ ]:
marca("S-05", "Diagnóstico de calidad en Bronze")
ts = lambda c: F.expr(f"try_cast({c} AS TIMESTAMP)")
reglas_asis = {
    "ID nulo o no numérico":            F.expr("try_cast(ID_Asistencia AS INT)").isNull(),
    "Trabajador nulo o no numérico":    F.expr("try_cast(ID_Trabajador AS INT)").isNull(),
    "Fecha de entrada inválida":        ts("Fecha_Entrada").isNull(),
    "Salida antes que entrada":         ts("Fecha_Salida") < ts("Fecha_Entrada"),
    "Estado fuera de catálogo":         ~F.trim("Estado_Asistencia").isin(ESTADOS_ASIS),
    "Estado no coincide con regla 09:41": (F.date_format(ts("Fecha_Entrada"), "HH:mm:ss") < LIMITE_PUNTUAL)
                                          != (F.trim("Estado_Asistencia") == "A tiempo"),
    "Espacios sobrantes en textos":     F.col("Estado_Asistencia") != F.trim("Estado_Asistencia"),
}
reglas_tarea = {
    "ID nulo o no numérico":            F.expr("try_cast(ID_Tarea AS INT)").isNull(),
    "Fecha de la tarea inválida":       F.expr("try_cast(Fecha AS DATE)").isNull(),
    "Entrega antes que inicio":         ts("Fecha_Entrega") < ts("Fecha_Inicio"),
    "Estado fuera de catálogo":         ~F.trim("Estado_Tarea").isin(ESTADOS_TAREA),
    "Completada sin fecha de entrega":  (F.trim("Estado_Tarea") == "Completada") & ts("Fecha_Entrega").isNull(),
    "En Progreso sin fecha de inicio":  (F.trim("Estado_Tarea") == "En Progreso") & ts("Fecha_Inicio").isNull(),
}
def contar(df, tabla, reglas, clave):
    filas = []
    dups = df.groupBy("_escala", clave).count().filter("count > 1").groupBy("_escala").agg(F.sum(F.col("count") - 1).alias("n"))
    dups = {r["_escala"]: int(r["n"]) for r in dups.collect()}
    for nombre, cond in reglas.items():
        for r in df.groupBy("_escala").agg(F.sum(F.when(cond, 1).otherwise(0)).alias("n")).collect():
            filas.append((tabla, nombre, r["_escala"], int(r["n"] or 0)))
    for e in ESCALAS:
        filas.append((tabla, f"Duplicado por {clave}", e, dups.get(e, 0)))
    return filas

diag = spark.createDataFrame(contar(bronze_asis, "asistencias", reglas_asis, "ID_Asistencia")
                             + contar(bronze_tarea, "tareas", reglas_tarea, "ID_Tarea"),
                             "tabla string, regla string, escala string, filas long")
diag_pv = diag.groupBy("tabla", "regla").pivot("escala", list(ESCALAS)).sum("filas").orderBy("tabla", "regla")
display(diag_pv)
diag.write.mode("overwrite").saveAsTable(f"{SILVER}.diagnostico_calidad")

## 4 · Capa **Silver**: limpieza y preparación — **CAPTURA S-06**
Decisiones de limpieza (se documentan en el informe):
1. `select` + `withColumnRenamed`: nombres en minúscula y sin mayúsculas mezcladas (`ID_Trabajador` → `id_trabajador`, …).
2. `trim` y normalización de estados; tipos correctos con `cast` (enteros, `timestamp`, `date`).
3. `na.drop` en columnas clave (id, trabajador, fecha): una fila sin clave no se puede usar.
4. `dropDuplicates` por (escala, id).
5. `filter`: se descartan fechas imposibles (salida antes que entrada, entrega antes que inicio) y estados fuera de catálogo.
6. `na.fill`: `observaciones` vacía → `""`; las banderas se calculan **antes** de rellenar, con la misma regla del Data Mart.
7. Columnas derivadas: mes, día de la semana, minutos de entrada, horas trabajadas, banderas de puntualidad y de tarea.
8. Las tareas con alerta (p. ej. *Completada sin fecha de entrega*) **no se borran**: el estado es el dato que usa el KPI;
   se marcan en `alerta_calidad` y se copian a una tabla de cuarentena para revisarlas.

In [ ]:
marca("S-06", "Silver: limpieza y preparación")
w_dup_a = Window.partitionBy("escala", "id_asistencia").orderBy("fecha_entrada")
silver_asis = (bronze_asis
    .select("ID_Asistencia", "ID_Trabajador", "Fecha_Entrada", "Fecha_Salida", "Fecha_Calculada", "Estado_Asistencia", "_escala")
    .withColumnRenamed("ID_Asistencia", "id_asistencia").withColumnRenamed("ID_Trabajador", "id_trabajador")
    .withColumnRenamed("Fecha_Entrada", "fecha_entrada").withColumnRenamed("Fecha_Salida", "fecha_salida")
    .withColumnRenamed("Fecha_Calculada", "fecha_dia").withColumnRenamed("Estado_Asistencia", "estado_asistencia")
    .withColumnRenamed("_escala", "escala")
    .withColumn("id_asistencia", F.expr("try_cast(trim(id_asistencia) AS INT)"))
    .withColumn("id_trabajador", F.expr("try_cast(trim(id_trabajador) AS INT)"))
    .withColumn("fecha_entrada", F.expr("try_cast(trim(fecha_entrada) AS TIMESTAMP)"))
    .withColumn("fecha_salida",  F.expr("try_cast(trim(fecha_salida) AS TIMESTAMP)"))
    .withColumn("fecha_dia",     F.expr("try_cast(trim(fecha_dia) AS DATE)"))
    .withColumn("estado_asistencia", F.trim("estado_asistencia"))
    .na.drop(subset=["id_asistencia", "id_trabajador", "fecha_entrada"])
    .dropDuplicates(["escala", "id_asistencia"])
    .filter(F.col("estado_asistencia").isin(ESTADOS_ASIS))
    .filter(F.col("fecha_salida").isNull() | (F.col("fecha_salida") >= F.col("fecha_entrada")))
    .withColumn("mes", F.date_format("fecha_entrada", "yyyy-MM"))
    .withColumn("dia_semana", F.date_format("fecha_entrada", "E"))
    .withColumn("minuto_entrada", F.hour("fecha_entrada") * 60 + F.minute("fecha_entrada"))
    .withColumn("horas_trabajadas", F.round((F.unix_timestamp("fecha_salida") - F.unix_timestamp("fecha_entrada")) / 3600, 2))
    .withColumn("es_puntual", F.when(F.col("estado_asistencia") == "A tiempo", 1).otherwise(0))
)

silver_tarea = (bronze_tarea
    .select("ID_Tarea", "ID_Trabajador", "ID_Proyecto", "Fecha", "Fecha_Inicio", "Fecha_Entrega", "Estado_Tarea", "Observaciones", "_escala")
    .withColumnRenamed("ID_Tarea", "id_tarea").withColumnRenamed("ID_Trabajador", "id_trabajador")
    .withColumnRenamed("ID_Proyecto", "id_proyecto").withColumnRenamed("Fecha", "fecha")
    .withColumnRenamed("Fecha_Inicio", "fecha_inicio").withColumnRenamed("Fecha_Entrega", "fecha_entrega")
    .withColumnRenamed("Estado_Tarea", "estado_tarea").withColumnRenamed("Observaciones", "observaciones")
    .withColumnRenamed("_escala", "escala")
    .withColumn("id_tarea",      F.expr("try_cast(trim(id_tarea) AS INT)"))
    .withColumn("id_trabajador", F.expr("try_cast(trim(id_trabajador) AS INT)"))
    .withColumn("id_proyecto",   F.expr("try_cast(trim(id_proyecto) AS INT)"))
    .withColumn("fecha",         F.expr("try_cast(trim(fecha) AS DATE)"))
    .withColumn("fecha_inicio",  F.expr("try_cast(trim(fecha_inicio) AS TIMESTAMP)"))
    .withColumn("fecha_entrega", F.expr("try_cast(trim(fecha_entrega) AS TIMESTAMP)"))
    .withColumn("estado_tarea",  F.trim("estado_tarea"))
    .withColumn("observaciones", F.trim("observaciones"))
    .na.drop(subset=["id_tarea", "id_trabajador", "fecha"])
    .dropDuplicates(["escala", "id_tarea"])
    .filter(F.col("estado_tarea").isin(ESTADOS_TAREA))
    .filter(F.col("fecha_entrega").isNull() | F.col("fecha_inicio").isNull() | (F.col("fecha_entrega") >= F.col("fecha_inicio")))
    # banderas con la regla del Data Mart (antes de rellenar nulos)
    .withColumn("completada_sin_obs", F.when((F.col("estado_tarea") == "Completada")
                & (F.col("observaciones").isNull() | (F.col("observaciones") == "")), 1).otherwise(0))
    .withColumn("fuera_de_plazo", F.when(F.upper("observaciones").contains("FUERA DE PLAZO"), 1).otherwise(0))
    .withColumn("alerta_calidad", F.when((F.col("estado_tarea") == "Completada") & F.col("fecha_entrega").isNull(),
                                         "Completada sin fecha de entrega")
                                   .when((F.col("estado_tarea") == "En Progreso") & F.col("fecha_inicio").isNull(),
                                         "En Progreso sin fecha de inicio"))
    .na.fill({"observaciones": ""})
    .withColumn("mes", F.date_format("fecha", "yyyy-MM"))
    .withColumn("horas_ejecucion", F.round((F.unix_timestamp("fecha_entrega") - F.unix_timestamp("fecha_inicio")) / 3600, 2))
)

silver_asis.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.asistencias")
silver_tarea.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.tareas")
spark.table(f"{SILVER}.tareas").filter("alerta_calidad IS NOT NULL") \
     .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.tareas_cuarentena")

spark.table(f"{SILVER}.asistencias").printSchema()
display(spark.table(f"{SILVER}.asistencias").filter("escala = 'base'").limit(5))
display(spark.table(f"{SILVER}.tareas").filter("escala = 'base'").limit(5))

## 5 · Conciliación Bronze ↔ Silver y cuarentena — **CAPTURA S-07**

In [ ]:
marca("S-07", "Conciliación Bronze vs Silver")
def conteo(t, nombre):
    return spark.table(t).groupBy(F.col("_escala" if "bronze" in t else "escala").alias("escala")).agg(F.count("*").alias(nombre))
conc = (conteo(f"{BRONZE}.asistencias", "bronze_asistencias")
        .join(conteo(f"{SILVER}.asistencias", "silver_asistencias"), "escala")
        .join(conteo(f"{BRONZE}.tareas", "bronze_tareas"), "escala")
        .join(conteo(f"{SILVER}.tareas", "silver_tareas"), "escala")
        .withColumn("descartadas_asistencias", F.col("bronze_asistencias") - F.col("silver_asistencias"))
        .withColumn("descartadas_tareas", F.col("bronze_tareas") - F.col("silver_tareas"))
        .orderBy(F.col("bronze_asistencias")))
display(conc)
print("Tareas en cuarentena (se conservan en Silver con alerta_calidad):")
display(spark.table(f"{SILVER}.tareas_cuarentena").groupBy("escala", "alerta_calidad").count().orderBy("escala", "alerta_calidad"))

## 6 · Capa **Gold**: indicadores — **CAPTURA S-08**
Mismas reglas del Data Mart (3.4.1): se suman numeradores y denominadores y luego se divide.
- **TPA** = asistencias "A tiempo" / asistencias · **ICT** = tareas Completadas sin observación / tareas
- **TDT** = tareas con "FUERA DE PLAZO" / tareas · **IEP** = 0,3·TPA + 0,5·ICT + 0,2·(100 − TDT)

In [ ]:
marca("S-08", "Gold: tablas de indicadores")
sa, st = spark.table(f"{SILVER}.asistencias"), spark.table(f"{SILVER}.tareas")

def kpi(df):
    return (df.withColumn("TPA", F.round(100 * F.col("puntuales") / F.col("asistencias"), 2))
              .withColumn("ICT", F.round(100 * F.col("sin_obs") / F.col("tareas"), 2))
              .withColumn("TDT", F.round(100 * F.col("fuera_plazo") / F.col("tareas"), 2))
              .withColumn("IEP", F.round(0.3 * 100 * F.col("puntuales") / F.col("asistencias")
                                         + 0.5 * 100 * F.col("sin_obs") / F.col("tareas")
                                         + 0.2 * (100 - 100 * F.col("fuera_plazo") / F.col("tareas")), 2)))

def gold(claves):
    a = sa.groupBy(*claves).agg(F.count("*").alias("asistencias"), F.sum("es_puntual").alias("puntuales"),
                                F.round(F.avg("horas_trabajadas"), 2).alias("horas_prom"))
    t = st.groupBy(*claves).agg(F.count("*").alias("tareas"), F.sum("completada_sin_obs").alias("sin_obs"),
                                F.sum("fuera_de_plazo").alias("fuera_plazo"))
    return kpi(a.join(t, claves, "full_outer").na.fill(0, ["asistencias", "puntuales", "tareas", "sin_obs", "fuera_plazo"]))

g_global = gold(["escala"])
g_mes    = gold(["escala", "mes"])
g_trab   = gold(["escala", "id_trabajador", "mes"])
g_trab_total = gold(["escala", "id_trabajador"]).withColumn(
    "ranking_iep", F.rank().over(Window.partitionBy("escala").orderBy(F.desc("IEP"))))

for nombre, df in (("kpi_global", g_global), ("kpi_mes", g_mes),
                   ("kpi_trabajador_mes", g_trab), ("kpi_trabajador", g_trab_total)):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.{nombre}")
    print(f"{GOLD}.{nombre}: {spark.table(f'{GOLD}.{nombre}').count()} filas")

display(spark.table(f"{GOLD}.kpi_global").orderBy("asistencias"))
display(spark.table(f"{GOLD}.kpi_mes").orderBy("escala", "mes"))

## 7 · Validación contra Power BI — **CAPTURA S-09**

In [ ]:
marca("S-09", "KPI Gold (escala base) vs Power BI")
b = spark.table(f"{GOLD}.kpi_global").filter("escala = 'base'").first().asDict()
val = pd.DataFrame([{"KPI": k, "Gold Databricks": float(b[k]), "Power BI": v,
                     "diferencia": round(float(b[k]) - v, 2), "coincide": abs(float(b[k]) - v) < 0.005}
                    for k, v in REF_POWER_BI.items()])
display(val)
print("Resultado:", "TODOS COINCIDEN" if val["coincide"].all() else "HAY DIFERENCIAS: revisar")
print("\nKPI por trabajador (escala base, 3 meses):")
display(spark.table(f"{GOLD}.kpi_trabajador").filter("escala = 'base'")
        .select("ranking_iep", "id_trabajador", "asistencias", "tareas", "TPA", "ICT", "TDT", "IEP").orderBy("ranking_iep", "id_trabajador"))

## 8 · Visualizaciones — **CAPTURA S-10**

In [ ]:
marca("S-10", "Visualizaciones de la capa Gold")
import matplotlib.pyplot as plt
AZUL, NARANJA, AQUA, AMARILLO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
TINTA, GRIS = "#3d3d3a", "#d6d5cf"
def estilo(ax):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(GRIS); ax.spines["bottom"].set_color(GRIS)
    ax.tick_params(colors=TINTA); ax.yaxis.grid(True, color="#ecebe6"); ax.set_axisbelow(True)

mes = spark.table(f"{GOLD}.kpi_mes").filter("escala = 'base'").orderBy("mes").toPandas()
trab = spark.table(f"{GOLD}.kpi_trabajador").filter("escala = 'base'").orderBy("IEP").toPandas()
ent = spark.table(f"{SILVER}.asistencias").filter("escala = 'base'").select("minuto_entrada", "estado_asistencia").toPandas()
glob = spark.table(f"{GOLD}.kpi_global").orderBy("asistencias").toPandas()

fig, ax = plt.subplots(2, 2, figsize=(13, 8.5))
# 1. KPI por mes
for k, c, m in (("TPA", AZUL, "o"), ("ICT", NARANJA, "s"), ("TDT", AQUA, "^"), ("IEP", AMARILLO, "D")):
    ax[0, 0].plot(mes["mes"], mes[k], color=c, marker=m, lw=2, ms=8, label=k)
    ax[0, 0].annotate(f"{mes[k].iloc[-1]:.1f}", (len(mes) - 1, mes[k].iloc[-1]), xytext=(8, 0),
                      textcoords="offset points", va="center", color=TINTA, fontsize=9)
ax[0, 0].set_title("KPI por mes · datos reales (582 filas)", loc="left", color=TINTA)
ax[0, 0].set_ylabel("%"); ax[0, 0].set_ylim(0, 105); ax[0, 0].legend(frameon=False, ncol=4, loc="lower center"); estilo(ax[0, 0])
# 2. IEP por trabajador
ax[0, 1].barh(trab["id_trabajador"].astype(str), trab["IEP"], color=AZUL, height=0.6)
for i, v in enumerate(trab["IEP"]):
    ax[0, 1].text(v + 0.5, i, f"{v:.1f}", va="center", color=TINTA, fontsize=9)
ax[0, 1].set_title("IEP por trabajador (ID) · jul–sep 2026", loc="left", color=TINTA)
ax[0, 1].set_xlabel("IEP"); ax[0, 1].set_xlim(0, 100); estilo(ax[0, 1])
# 3. Distribución de hora de entrada
lim = 9 * 60 + 41
for est, c in (("A tiempo", AZUL), ("Tardanza", NARANJA)):
    ax[1, 0].hist(ent.loc[ent["estado_asistencia"] == est, "minuto_entrada"], bins=range(480, 621, 5),
                  color=c, label=est, edgecolor="white", linewidth=1)
ax[1, 0].axvline(lim, color=TINTA, ls="--", lw=1); ax[1, 0].text(lim + 1, ax[1, 0].get_ylim()[1] * 0.92, "09:41", color=TINTA, fontsize=9)
ax[1, 0].set_xticks(range(480, 621, 20)); ax[1, 0].set_xticklabels([f"{m // 60:02d}:{m % 60:02d}" for m in range(480, 621, 20)])
ax[1, 0].set_title("Hora de entrada (asistencias reales)", loc="left", color=TINTA)
ax[1, 0].set_ylabel("asistencias"); ax[1, 0].legend(frameon=False); estilo(ax[1, 0])
# 4. KPI por escala
import numpy as np
x = np.arange(4); wb = 0.26
for i, (esc, c) in enumerate(zip(glob["escala"], (AZUL, NARANJA, AQUA))):
    vals = [glob.loc[i, k] for k in ("TPA", "ICT", "TDT", "IEP")]
    ax[1, 1].bar(x + (i - 1) * wb, vals, wb * 0.92, color=c, label=f"{int(glob.loc[i, 'asistencias']):,} filas".replace(",", " "))
ax[1, 1].set_xticks(x); ax[1, 1].set_xticklabels(["TPA", "ICT", "TDT", "IEP"])
ax[1, 1].set_title("KPI Gold por escala de datos", loc="left", color=TINTA)
ax[1, 1].set_ylabel("%"); ax[1, 1].set_ylim(0, 105); ax[1, 1].legend(frameon=False, ncol=3, loc="upper right"); estilo(ax[1, 1])
plt.tight_layout()
plt.savefig(f"{RUTA_SALIDA}/graficos_gold.png", dpi=150)
plt.show()

## 9 · Persistencia: tablas Delta, comentarios e historial — **CAPTURA S-11**

In [ ]:
marca("S-11", "Persistencia en Unity Catalog")
comentarios = {
    f"{BRONZE}.asistencias": "Bronze: asistencias crudas (texto) de las 3 escalas, con archivo de origen y fecha de ingesta",
    f"{BRONZE}.tareas": "Bronze: tareas crudas (texto) de las 3 escalas, con archivo de origen y fecha de ingesta",
    f"{SILVER}.asistencias": "Silver: asistencias limpias y tipadas, con mes, minuto de entrada, horas y bandera de puntualidad",
    f"{SILVER}.tareas": "Silver: tareas limpias y tipadas, con banderas completada_sin_obs, fuera_de_plazo y alerta_calidad",
    f"{SILVER}.tareas_cuarentena": "Silver: tareas con alerta de calidad para revisión (no se borran de Silver)",
    f"{SILVER}.diagnostico_calidad": "Silver: conteo de filas que incumplen cada regla de calidad en Bronze, por escala",
    f"{GOLD}.kpi_global": "Gold: TPA, ICT, TDT e IEP por escala",
    f"{GOLD}.kpi_mes": "Gold: TPA, ICT, TDT e IEP por escala y mes",
    f"{GOLD}.kpi_trabajador_mes": "Gold: TPA, ICT, TDT e IEP por escala, trabajador (ID) y mes",
    f"{GOLD}.kpi_trabajador": "Gold: TPA, ICT, TDT e IEP por escala y trabajador (ID), con ranking por IEP",
}
for t, c in comentarios.items():
    spark.sql(f"COMMENT ON TABLE {t} IS '{c}'")
filas = []
for esq in (BRONZE, SILVER, GOLD):
    for r in spark.sql(f"SHOW TABLES IN {esq}").collect():
        t = f"{esq}.{r['tableName']}"
        d = spark.sql(f"DESCRIBE DETAIL {t}").first()
        filas.append((esq.split(".")[1], r["tableName"], d["format"], spark.table(t).count(), d["sizeInBytes"], comentarios.get(t, "")))
display(spark.createDataFrame(filas, "capa string, tabla string, formato string, filas long, bytes long, comentario string"))
print("Historial Delta de la tabla Gold kpi_trabajador_mes:")
display(spark.sql(f"DESCRIBE HISTORY {GOLD}.kpi_trabajador_mes").select("version", "timestamp", "operation", "operationMetrics"))

In [ ]:
# Exporta la capa Gold de la escala base como CSV (evidencia para el repositorio)
for t in ("kpi_global", "kpi_mes", "kpi_trabajador_mes", "kpi_trabajador"):
    pdf = spark.table(f"{GOLD}.{t}").toPandas()
    pdf.to_csv(f"{RUTA_SALIDA}/gold_{t}.csv", index=False)
spark.table(f"{SILVER}.diagnostico_calidad").toPandas().to_csv(f"{RUTA_SALIDA}/diagnostico_calidad.csv", index=False)
display(dbutils.fs.ls(RUTA_SALIDA))

## 10 · Dashboard de Databricks sobre la capa Gold — **CAPTURA S-12**
La celda crea (o actualiza) el dashboard **"BASAV · Monitoreo de empleados (Gold)"** con el SDK de Databricks y lo publica
con el SQL warehouse del espacio de trabajo. Abrir el enlace que imprime y capturarlo.
Si la creación automática falla, el mensaje de error queda impreso y el dashboard se arma a mano en **Dashboards → Create dashboard**
con las mismas consultas (ver `datasets` abajo).

In [ ]:
marca("S-12", "Dashboard AI/BI sobre Gold")
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
NOMBRE_DASH = "BASAV · Monitoreo de empleados (Gold)"

datasets = [
    {"name": "d_global", "displayName": "KPI global (base)",
     "queryLines": [f"SELECT TPA, ICT, TDT, IEP, asistencias, tareas FROM {GOLD}.kpi_global WHERE escala = 'base'"]},
    {"name": "d_mes", "displayName": "KPI por mes (base)",
     "queryLines": [f"SELECT mes, kpi, valor FROM {GOLD}.kpi_mes UNPIVOT (valor FOR kpi IN (TPA, ICT, TDT, IEP)) WHERE escala = 'base'"]},
    {"name": "d_trab", "displayName": "IEP por trabajador (base)",
     "queryLines": [f"SELECT CONCAT('ID ', CAST(id_trabajador AS STRING)) AS trabajador, IEP, TPA, ICT, TDT FROM {GOLD}.kpi_trabajador WHERE escala = 'base'"]},
    {"name": "d_escala", "displayName": "KPI por escala",
     "queryLines": [f"SELECT CONCAT(CAST(asistencias AS STRING), ' filas') AS escala, kpi, valor FROM {GOLD}.kpi_global UNPIVOT (valor FOR kpi IN (TPA, ICT, TDT, IEP))"]},
    {"name": "d_calidad", "displayName": "Cuarentena por escala",
     "queryLines": [f"SELECT escala, alerta_calidad, COUNT(*) AS tareas FROM {SILVER}.tareas_cuarentena GROUP BY ALL"]},
]

def contador(nombre, campo, titulo, x):
    return {"widget": {"name": nombre,
            "queries": [{"name": "main_query", "query": {"datasetName": "d_global",
                         "fields": [{"name": campo, "expression": f"`{campo}`"}], "disaggregated": True}}],
            "spec": {"version": 2, "widgetType": "counter",
                     "encodings": {"value": {"fieldName": campo, "displayName": titulo}},
                     "frame": {"showTitle": True, "title": titulo}}},
            "position": {"x": x, "y": 1, "width": 1, "height": 3}}

def grafico(nombre, tipo, ds, x, y, color, titulo, pos, x_tipo="categorical"):
    campos = [{"name": x, "expression": f"`{x}`"}, {"name": y, "expression": f"`{y}`"}]
    enc = {"x": {"fieldName": x, "scale": {"type": x_tipo}, "displayName": x},
           "y": {"fieldName": y, "scale": {"type": "quantitative"}, "displayName": y}}
    if color:
        campos.append({"name": color, "expression": f"`{color}`"})
        enc["color"] = {"fieldName": color, "scale": {"type": "categorical"}, "displayName": color}
    return {"widget": {"name": nombre,
            "queries": [{"name": "main_query", "query": {"datasetName": ds, "fields": campos, "disaggregated": True}}],
            "spec": {"version": 3, "widgetType": tipo, "encodings": enc, "frame": {"showTitle": True, "title": titulo}}},
            "position": pos}

texto = {"widget": {"name": "titulo", "multilineTextboxSpec": {"lines": [
    "## BASAV · Monitoreo de empleados — capa Gold\n",
    "Datos reales del Data Mart (582 asistencias y 582 tareas, jul–sep 2026). Fuente: tablas Delta `workspace.mon_gold`."]}},
    "position": {"x": 0, "y": 0, "width": 6, "height": 1}}

layout = [texto,
          contador("c_tpa", "TPA", "TPA · puntualidad (%)", 0), contador("c_ict", "ICT", "ICT · tareas sin observación (%)", 1),
          contador("c_tdt", "TDT", "TDT · tareas fuera de plazo (%)", 2), contador("c_iep", "IEP", "IEP · índice de productividad", 3),
          contador("c_asis", "asistencias", "Asistencias", 4), contador("c_tar", "tareas", "Tareas", 5),
          grafico("g_mes", "line", "d_mes", "mes", "valor", "kpi", "KPI por mes", {"x": 0, "y": 4, "width": 3, "height": 6}),
          grafico("g_trab", "bar", "d_trab", "trabajador", "IEP", None, "IEP por trabajador", {"x": 3, "y": 4, "width": 3, "height": 6}),
          grafico("g_esc", "bar", "d_escala", "kpi", "valor", "escala", "KPI por escala de datos (582 / 5 000 / 10 000)",
                  {"x": 0, "y": 10, "width": 3, "height": 6}),
          grafico("g_cal", "bar", "d_calidad", "escala", "tareas", "alerta_calidad", "Tareas en cuarentena por escala",
                  {"x": 3, "y": 10, "width": 3, "height": 6})]
dash_json = json.dumps({"datasets": datasets,
                        "pages": [{"name": "resumen", "displayName": "Resumen", "layout": layout}]})

almacenes = list(w.warehouses.list())
print("SQL warehouses:", [(x.name, x.id, str(x.state)) for x in almacenes])
wid = almacenes[0].id if almacenes else None
carpeta = os.path.dirname(NOTEBOOK) if NOTEBOOK.startswith("/") else None

from databricks.sdk.service.dashboards import Dashboard
existente = next((d for d in w.lakeview.list() if d.display_name == NOMBRE_DASH), None)
obj = Dashboard(display_name=NOMBRE_DASH, serialized_dashboard=dash_json, warehouse_id=wid, parent_path=carpeta)
if existente:
    obj.parent_path = None
    dash = w.lakeview.update(existente.dashboard_id, dashboard=obj)
    print("Dashboard actualizado")
else:
    dash = w.lakeview.create(dashboard=obj)
    print("Dashboard creado")
try:
    w.lakeview.publish(dash.dashboard_id, warehouse_id=wid, embed_credentials=True)
    print("Dashboard publicado")
except Exception as ex:
    print("No se pudo publicar (queda en borrador):", str(ex)[:300])
url = f"{w.config.host.rstrip('/')}/dashboardsv3/{dash.dashboard_id}"
print("ID:", dash.dashboard_id)
displayHTML(f'<a href="{url}" target="_blank">Abrir el dashboard: {NOMBRE_DASH}</a>')

## Observaciones (se llenan con lo que muestran las salidas)
- La capa Gold de la escala base debe dar los mismos KPI que Power BI (S-09).
- Las filas descartadas por la limpieza se ven en S-07; las alertas de calidad en la cuarentena.
- Límites: las escalas 5 000 y 10 000 son sintéticas (generadas a partir de los trabajadores reales);
  sus KPI no describen a trabajadores reales y se usan solo para probar el pipeline con más volumen.